In [ ]:
### Part A - Preparation od DataFames < Clean, Correction and Imputation > ###
import pandas as pd
import numpy as np
#1 Import all files together (Reviews, Listings and Calendar)
dfs = {
    'Listings': pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Listings.csv'),
    'Calendar': pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/calendar.csv'),
    'Reviews': pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Reviews.csv')
}

In [ ]:
#2 Count of rows and columns for each DataFrame
for name, df in dfs.items():
  print(name)
  print(f"Shape ( Rows and columns): {df.shape}\n")

#3 First 5 rows for each DataFrame
  print('First 5 rows:')
  print(df.head(5))
  print('\n' + "-"*20)

#4 Columns with missing values
  print('Columns with missing values:')
  missing= df.isnull().sum()
  print(missing[missing > 0] if missing.sum() > 0 else "N/A")
  print("\n" + '-'*20)

#5 Descriptive Statistics
  print("Descriptive Statistics:")
  print(df.describe().round(2))
  print("\n\n")

Listings
Shape ( Rows and columns): (12955, 75)

First 5 rows:
         id                          listing_url     scrape_id last_scraped  \
0   10595.0   https://www.airbnb.com/rooms/10595  2.023090e+13   21/09/2023   
1   10990.0   https://www.airbnb.com/rooms/10990  2.023090e+13   21/09/2023   
2   10993.0   https://www.airbnb.com/rooms/10993  2.023090e+13   21/09/2023   
3   10995.0   https://www.airbnb.com/rooms/10995  2.023090e+13   21/09/2023   
4  724485.0  https://www.airbnb.com/rooms/724485  2.023090e+13   21/09/2023   

        source                                               name  \
0  city scrape  Condo in Athens · ★4.83 · 3 bedrooms · 5 beds ...   
1  city scrape  Rental unit in Athens · ★4.80 · 1 bedroom · 1 ...   
2  city scrape  Rental unit in Athens · ★4.84 · Studio · 2 bed...   
3  city scrape  Rental unit in Athens · ★4.80 · 1 bedroom · 2 ...   
4  city scrape  Rental unit in Athens · ★4.80 · 1 bedroom · 1 ...   

                                         descri

In [ ]:
# Correction of the price type in Listings and Calendar DataFrames
for Data in ['Listings', 'Calendar']:
   dfs[Data]['price'] = dfs[Data]['price'].astype(str)\
                            .str.replace('$', '', regex=False)\
                            .str.replace(',', '', regex=False)\
                            .astype(float)

# Corrections of rate type for response and acceptance rate columns respectively
rate_columns = ['host_response_rate', 'host_acceptance_rate']

for col in rate_columns:
  dfs['Listings'][col]=dfs['Listings'][col].astype(str).str.replace('%', '', regex=False)
  dfs['Listings'][col]=pd.to_numeric(dfs['Listings'][col],errors='coerce')

print('Host Response Rate type:', dfs['Listings']['host_response_rate'].dtype)
print('Host Acceptance Rate type:', dfs['Listings']['host_acceptance_rate'].dtype)

Host Response Rate type: float64
Host Acceptance Rate type: float64


In [ ]:
# Remove some useless columns from Listings DataFrame
# Dropping 100% empty columns ('neighbourhood_group_cleansed', 'bathrooms', 'calendar_updated')
# Dropping technical metadata & URLs ('scrape_id', 'host_thumbnail_url', 'host_picture_url', 'calendar_last_scraped')

completely_empty_cols = ['neighbourhood_group_cleansed', 'bathrooms', 'calendar_updated']
useless_metadata_cols = ['scrape_id', 'host_thumbnail_url', 'host_picture_url', 'calendar_last_scraped']
cols_to_drop = [col for col in completely_empty_cols + useless_metadata_cols if col in dfs['Listings'].columns]
dfs['Listings'].drop(columns=cols_to_drop, axis = 1, inplace=True)
dfs['Listings'].columns.tolist()

# Imputing missing categorical data. Using 'Unknown' placeholder for text columns
# Mode Imputation (most frequent value) for operational metrics like response time
# Left ('bedrooms', 'beds', 'host_response_rate', 'review_scores_rating') which contained Nan values to avoid distorting realistic measurements.

filled_columns = ['license','host_about']
for col in filled_columns:
  if col in dfs['Listings'].columns:
    dfs['Listings'][col]= dfs['Listings'][col].fillna('Unknown')

most_frequent = dfs['Listings']['host_response_time'].mode()[0]
dfs['Listings']['host_response_time']=dfs['Listings']['host_response_time'].fillna(most_frequent)

In [ ]:
# Frequency booking calculation
dfs['Listings']['id']=dfs['Listings']['id'].astype(int) # Change the type of 'id' as integer
merged_dfs = pd.merge(dfs['Calendar'], dfs['Listings'][['id','name']],
                   left_on ='listing_id',
                   right_on='id')

grouped = merged_dfs.groupby(['id','name','available']).size()
freq_booking = grouped.loc[:,:,'f'].sort_values(ascending=False)
print('Top 10 Most Booked Listings (Days Booked):')
freq_booking.head(10).to_frame(name='days_booked')

Top 10 Most Booked Listings (Days Booked):


,,days_booked
id,name,
31155,Rental unit in Athens · 1 bedroom · 1 bed · 1 shared bath,365
54400247,Rental unit in Athina · 1 bedroom · 1 bed · 1 bath,365
21909590,Rental unit in Athina · ★4.97 · 1 bedroom · 2 beds · 1 bath,365
21933793,Rental unit in Athina · ★4.89 · 3 bedrooms · 5 beds · 1.5 baths,365
21934137,Rental unit in Athina · ★4.91 · 1 bedroom · 1 bed · 1.5 baths,365
21939347,Home in Athina · Studio · 1 bed · 1.5 baths,365
21495621,Rental unit in Athina · 1 bedroom · 1 bed · 1 bath,365
21497405,Rental unit in Athina · ★5.0 · Studio · 1 bed · 1 bath,365
21547069,Rental unit in Glifada · 1 bedroom · 2 beds · 1 bath,365


In [ ]:
# Average price per listing_place
merged_dfs = pd.merge(dfs['Calendar'], dfs['Listings'][['id','name']],
                   left_on ='listing_id',
                   right_on='id')
#Filtered out listings with prices >= 2000 to exclude extreme outliers/luxury properties or data entry errors
filtered_dfs = merged_dfs[merged_dfs['price'] < 2000]
avg_price = filtered_dfs.groupby(['id','name'])['price'].mean().round(2)
highest_avg_price = avg_price.sort_values(ascending=False)
print('Average Price of Top 10 Listings (< $2000 Filter Applied):')
highest_avg_price.head(10).to_frame(name='avg_price_euro')


Average Price of Top 10 Listings (< $2000 Filter Applied):


,,avg_price_euro
id,name,
28042027,Aparthotel in Athens · ★5.0 · 6 bedrooms · 9 beds · 6.5 baths,1650.97
51282041,Rental unit in Athina · ★4.71 · 2 bedrooms · 3 beds · 1 bath,1604.59
41556060,Condo in Athina · ★5.0 · 9 bedrooms · 9 beds · 12 baths,1567.39
22462133,Rental unit in Athina · ★4.90 · 2 bedrooms · 2 beds · 1.5 baths,1500.00
51421032,Serviced apartment in Athina · ★4.87 · 6 bedrooms · 6 beds · 6 baths,1364.44
45376183,Serviced apartment in Athina · ★4.93 · 2 bedrooms · 2 beds · 2.5 baths,1194.62
35394051,Boutique hotel in Athina · Studio · 1 bed · 1 shared bath,1180.00
54402931,Condo in Athina · ★5.0 · 7 bedrooms · 18 beds · 6 baths,1146.71
37581767,Rental unit in Athina · ★4.80 · 2 bedrooms · 2 beds · 1 bath,1141.75


In [ ]:
# Top and the fewest reviews for Listings

listings_fix_columns = dfs['Listings'][['id', 'name']].rename(columns={'id': 'listing_id'})
dfs['Reviews']['listing_id'] = dfs['Reviews']['listing_id'].astype(float).astype(int)
listings_fix_columns['listing_id'] = listings_fix_columns['listing_id'].astype(float).astype(int)
freq_merged_dfs = pd.merge(dfs['Reviews'], listings_fix_columns, on='listing_id')
reviews_per_listing = freq_merged_dfs.groupby(['listing_id', 'name']).size()
sorted_reviews = reviews_per_listing.sort_values(ascending=False)
print('Top 5 listings with most reviews:')
print(sorted_reviews.head(5))

print("\n" + "-"*10)
print('The 5 listings with the fewest reviews')
print(sorted_reviews.tail(5))

Top 5 listings with most reviews:
listing_id  name                                                         
1177492     Earthen home in Athens · ★4.73 · 1 bedroom · 1 bed · 1 bath      865
3431705     Rental unit in Athens · ★4.84 · 3 bedrooms · 4 beds · 3 baths    856
13553080    Rental unit in Athens · ★4.85 · 2 bedrooms · 3 beds · 1 bath     778
14583913    Rental unit in Athina · ★4.68 · 1 bedroom · 2 beds · 1 bath      764
5025556     Rental unit in Athina · ★4.85 · 1 bedroom · 1 bed · 1 bath       731
dtype: int64

----------
The 5 listings with the fewest reviews
listing_id          name                                                       
973758000000000000  Rental unit in Menemeni · ★New · 1 bedroom · 1 bed · 1 bath    1
973704000000000000  Rental unit in Athina · ★New · 1 bedroom · 1 bed · 1 bath      1
973091000000000000  Rental unit in Athina · ★New · 1 bedroom · 1 bed · 1 bath      1
973035000000000000  Rental unit in Athina · ★New · Studio · 1 bed · 1 bath         1
973

In [15]:
#Frequency of the reviewing  per Listing
freq_merged_dfs['date'] = pd.to_datetime(freq_merged_dfs['date'])
reviews_by_year = freq_merged_dfs.groupby(freq_merged_dfs['date'].dt.year).size()

print("\n Frequency of reviews per Year:")
print(reviews_by_year)



 Frequency of reviews per Year:
date
2010         2
2011        30
2012       240
2013      1094
2014      3198
2015      7789
2016     16274
2017     34850
2018     64135
2019     98257
2020     41461
2021     77738
2022    147446
2023    141543
dtype: int64


In [ ]:
#Frequency of the reviewing  per Listing
freq_merged_dfs['date'] = pd.to_datetime(freq_merged_dfs['date'],format='%d/%m/%Y')
reviews_by_year = freq_merged_dfs.groupby(freq_merged_dfs['date'].dt.year).size()

print("\n Frequency of reviews per Year:")
print(reviews_by_year)


 Frequency of reviews per Year:
date
2010         2
2011        30
2012       240
2013      1094
2014      3198
2015      7789
2016     16274
2017     34850
2018     64135
2019     98257
2020     41461
2021     77738
2022    147446
2023    141543
dtype: int64


In [ ]:
# Applied .clip(upper=365) across both Listings and Calendar DataFrames.
dfs['Calendar']['minimum_nights'] = dfs['Calendar']['minimum_nights'].clip(upper=365)
dfs['Calendar']['maximum_nights'] = dfs['Calendar']['maximum_nights'].clip(upper=365)
dfs['Listings']['minimum_nights'] = dfs['Listings']['minimum_nights'].clip(upper=365)
dfs['Listings']['maximum_nights'] = dfs['Listings']['maximum_nights'].clip(upper=365)

print('Check limit of max booking days:')
print("Listings Max Minimum Nights:", dfs['Listings']['minimum_nights'].max())
print("Calendar Max Maximum Nights:", dfs['Calendar']['maximum_nights'].max())

Check limit of max booking days:
Listings Max Minimum Nights: 365
Calendar Max Maximum Nights: 365


In [ ]:
## Creation of copies for clean export
listings_clean = dfs['Listings'].copy()
new_calendar = dfs['Calendar'].copy()
listings_clean.to_csv(r'/content/drive/MyDrive/Colab Notebooks/listings_clean.csv', index=False)
new_calendar.to_csv(r'/content/drive/MyDrive/Colab Notebooks/new_calendar.csv', index=False)

In [ ]:
## Part B - Analysis with the new DataFrames ##
##1 Most profitable areas in Greece ##

booked_days = dfs['Calendar'][dfs['Calendar']['available']=='f']

##1 Total earnigs per Listing ##
# Calculating total earnings per listing. Ensuring matching data types (str)
listings_clean = listings_clean.rename(columns={'id': 'listing_id'})
revenue_per_listing = booked_days.groupby('listing_id')['price'].sum().reset_index()
revenue_per_listing.columns = ['listing_id', 'total_revenue']

listings_clean['listing_id'] = listings_clean['listing_id'].astype(str)
revenue_per_listing['listing_id'] = revenue_per_listing['listing_id'].astype(str)

listings_revenue= pd.merge(listings_clean,revenue_per_listing, on = 'listing_id', how = 'left')
listings_revenue['total_revenue']=listings_revenue['total_revenue'].fillna(0)
# Avg revenue per area
avg_areas_revenue = listings_revenue.groupby('neighbourhood_cleansed')['total_revenue'].mean().round(0)
top_10_areas = avg_areas_revenue.sort_values(ascending=False).head(10)

#Top 10 areas per earnings
print('Top 10 profitable areas:')
top_10_areas.round(2).apply(lambda x: "{:,.0f}".format(x)).to_frame(name='avg_revenue')

Top 10 profitable areas:


,avg_revenue
neighbourhood_cleansed,
ΣΤΑΘΜΟΣ ΛΑΡΙΣΗΣ,"83,813"
ΠΕΔΙΟ ΑΡΕΩΣ,"66,578"
ΘΗΣΕΙΟ,"38,419"
ΚΕΡΑΜΕΙΚΟΣ,"37,970"
ΑΚΡΟΠΟΛΗ,"34,472"
ΚΟΛΩΝΑΚΙ,"33,626"
ΝΕΑ ΚΥΨΕΛΗ,"32,570"
ΖΑΠΠΕΙΟ,"30,857"
ΣΤΑΔΙΟ,"30,239"


In [ ]:
## 2 Total reviews by Host ##

# Count the total reviews
# Grouping reviews by 'listing_id' to get total counts per property before aggregating by Host.
reviews_count = dfs['Reviews'].groupby('listing_id').size().reset_index(name='total_reviews')
listings_clean['listing_id'] = listings_clean['listing_id'].astype(str)
reviews_count['listing_id'] = reviews_count['listing_id'].astype(str)

listings_reviews= pd.merge(listings_clean,reviews_count, left_on='listing_id', right_on='listing_id', how= 'left')

listings_reviews['total_reviews'] = listings_reviews['total_reviews'].fillna(0)
host_reviews = listings_reviews.groupby('host_id')['total_reviews'].sum()
top_10_hosts = host_reviews.sort_values(ascending=False).head(10)

print('Top 10 hosts by reviews:')
top_10_hosts.apply(lambda x: '{:,.0f}'.format(x)).to_frame(name='total_reviews')

Top 10 hosts by reviews:


,total_reviews
host_id,
112527018,"13,505"
90390850,"11,017"
92310506,"7,694"
20104194,"5,749"
123074489,"4,707"
32601523,"4,143"
245702882,"3,502"
748818,"3,218"
22227167,"3,163"


In [ ]:

## 3 Pricing Strategy Efficiency ##
# Assessing pricing efficiency using revenue per day available.
available_days = dfs['Calendar'][dfs['Calendar']['available']=='t']
days_count = available_days.groupby("listing_id").size().reset_index(name='days_available')
days_count['listing_id'] = days_count['listing_id'].astype(str)

pricing_strategy = pd.merge(listings_clean, days_count, on='listing_id',  how = 'left')
pricing_strategy['days_available'] = pricing_strategy['days_available']
pricing_strategy['days_available'] = pricing_strategy['days_available'].fillna(0)

revenue_info=listings_revenue[['listing_id','total_revenue']].copy()
revenue_info['listing_id'] = revenue_info['listing_id'].astype(str)
pricing_strategy = pd.merge(pricing_strategy, revenue_info, on = 'listing_id', how='left')

## Filter for active listings (days_available > 30)
pricing_strategy['revenue_per_day'] = np.where(
    pricing_strategy['days_available'] > 0,
    pricing_strategy['total_revenue'] / pricing_strategy['days_available'],
    0
)

valid_listings = pricing_strategy[pricing_strategy['days_available'] > 30].copy()
valid_listings['revenue_per_day'] = valid_listings['total_revenue'] / valid_listings['days_available']
top_10_efficient_listings = valid_listings.sort_values(by='revenue_per_day', ascending=False)
top_10_efficient_listings[['listing_id', 'days_available', 'total_revenue', 'revenue_per_day']]\
    .reset_index(drop=True)\
    .head(10)\
    .style.format({
        'days_available': '{:,.0f}',
        'total_revenue': '{:,.2f} €',
        'revenue_per_day': '{:,.2f} €'
    })


,listing_id,days_available,total_revenue,revenue_per_day
0,32463338,111,"18,990,560.00 €","171,086.13 €"
1,48623239,31,"2,037,610.00 €","65,729.35 €"
2,38368469,31,"1,621,746.00 €","52,314.39 €"
3,38367574,39,"1,680,605.00 €","43,092.44 €"
4,45445654,40,"1,680,434.00 €","42,010.85 €"
5,35461470,54,"1,678,215.00 €","31,078.06 €"
6,35478322,55,"1,678,420.00 €","30,516.73 €"
7,33816859,57,"1,666,435.00 €","29,235.70 €"
8,35399104,57,"1,658,520.00 €","29,096.84 €"
9,35536596,66,"1,676,675.00 €","25,404.17 €"
